# 作业 3：构建 NDArray 库

本次作业将构建一个简单的底层库，用于实现大多数深度学习系统所依赖的处理功能：n 维数组（即 NDArray）。此前你主要使用 NumPy 完成这类操作；本次作业将带你开发一个功能有限的 NumPy 变体，同时支持 CPU 和 GPU 后端。与 NumPy（甚至 PyTorch 等变体）不同，你不会直接调用现成的高性能矩阵乘法或其他操作实现，而是要自己编写性能尚可的实现（以某种衡量方式来说，只比高优化库慢 2～3 倍；这已经远好于可能慢 100 倍的朴素实现）。最终，这个类会集成到 `needle` 中；但本次作业只需关注 ndarray 模块，因为测试也只涉及这一部分。

**需要 HW1/HW2 的代码吗？** 不需要。HW3 的测试只会覆盖 NDArray 后端（`python/needle/backend_ndarray/` 和 `src/`），因此可以保留其余 `python/needle/` 文件的原样；HW4 会说明需要沿用哪些文件。**注意：** HW3 的 [`python/needle/__init__.py`](https://github.com/dlsyscourse/hw3/blob/main/python/needle/__init__.py) 和 [`python/needle/autograd.py`](https://github.com/dlsyscourse/hw3/blob/main/python/needle/autograd.py) 包含新的后端选择代码，与 HW2 中的版本不同。

**CPU 与 GPU 运行环境：** 为避免耗尽 Colab 有限的 GPU 资源，Part 1～5 请使用 CPU 运行环境；CUDA 部分（Part 6～9）请切换到 GPU 运行环境（**Runtime → Change runtime type → T4 GPU**）。切换运行环境后会启动一台全新的机器，因此 Part 6 前的章节会说明如何重新运行设置并重新构建。

In [ ]:
# Code to set up the assignment
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/
!mkdir -p 10714
%cd /content/drive/MyDrive/10714
# skip the clone if it's already there (e.g. when re-running setup after switching runtimes)
![ -d hw3 ] || git clone https://github.com/dlsys10714/hw3.git
%cd /content/drive/MyDrive/10714/hw3

!pip3 install --upgrade --no-deps git+https://github.com/dlsys10714/mugrade.git
!pip3 install pybind11

In [ ]:
# REQUIRED FOR MUGRADE
MY_API_KEY = "<FILL YOUR API KEY HERE>"
HW3_NAME = "hw3"

In [ ]:
!make

`make` 命令会读取当前目录中的 Makefile。Makefile 包含定义构建目标（例如可执行文件或库）的规则。对于指定的每个目标，`make` 会检查目标文件及其依赖文件（例如 `.c`、`.cpp` 或 `.h` 文件）的时间戳。如果某个依赖文件最近有过修改，就需要重新构建目标。

每次修改 `.cc` 或 `.cu` 文件后，都需要重新运行 `make` 再进行测试；下文的 C++/CUDA 部分会在测试前提供 `!make` 单元格。修改 [`ndarray.py`](https://github.com/dlsyscourse/hw3/blob/main/python/needle/backend_ndarray/ndarray.py) 则不需要重新构建。

## 熟悉 NDArray 类

开始本次作业前，请先熟悉作为作业起点提供的 `NDArray` 类（位于 [`python/needle/backend_ndarray/ndarray.py`](https://github.com/dlsyscourse/hw3/blob/main/python/needle/backend_ndarray/ndarray.py)）。代码篇幅不长（约 500 行，其中不少是为待实现函数提供的注释）。

NDArray 类本质上是一个 Python 封装，用于处理通用的 n 维数组。回想一下，这类数组通常在内部以浮点数向量的形式存储，例如：

```c++
float data[size];
```

数组各维度的访问方式则由其他字段（例如 shape、strides 等）共同决定，这些字段描述了这个“一维”数组如何映射到 n 维结构。为了获得合理的运行速度，原始操作（例如加法、二元运算，以及矩阵乘法等结构化操作）都需要用 C++ 等原生语言实现（也可能调用 CUDA）。不过，转置、广播、矩阵切片等许多操作，只需调整数组的高层结构（例如 strides）即可完成。

NDArray 类的设计理念是：所有处理数组结构的逻辑都用 Python 编写。只有实际操作底层一维数据的低层代码，以及管理这些数据（例如在 GPU 上分配内存）的代码，才使用 C++。随着作业的推进，你会逐渐理解这种分工。通常来说，凡是 Python 能完成的工作，就由 Python 完成，即使这会带来一些效率损失；例如，我们会大量调用 `.compact()`（复制内存），以简化底层 C++ 实现。

NDArray 类中有五个需要熟悉的字段（实际成员名前都有下划线，例如 `_handle`、`_strides` 等；其中一部分会通过公开属性暴露。编写代码时可以直接使用带下划线的内部成员）：

1. `device`：`BackendDevice` 类型的对象，是对底层设备后端（例如 CPU 或 CUDA）的简单封装。
2. `handle`：保存数组底层内存的类对象，分配类型为 `device.Array()`。分配过程已在提供的代码（具体来说是 `NDArray.make` 函数）中完成，不需要自行调用。
3. `shape`：元组，指定数组每个维度的大小。
4. `strides`：元组，指定数组每个维度的步长。
5. `offset`：整数，表示数组在底层 `device.Array` 内存中的起始位置。记录它可以方便地指向已有内存，而不必跟踪分配过程。

通过调整这些字段，纯 Python 代码就能完成许多数组操作，例如维度置换（即转置）和广播。对于底层数组操作，`device` 类提供了若干以 C++ 实现的方法。

还有几点需要注意：

* 内部可以使用任何高效方式，以“设备”后端处理数组数据。例如，可以使用 NumPy 数组，但不直接用 `numpy.ndarray` 表示 n 维数组，而是只用 NumPy 表示一维数据，再调用相应 NumPy 方法实现底层内存操作。这正是 [`ndarray_backend_numpy.py`](https://github.com/dlsyscourse/hw3/blob/main/python/needle/backend_ndarray/ndarray_backend_numpy.py) 的做法：它提供了一个使用 NumPy 实现所有操作的参考后端。你可以用它调试自己的原生 CPU 和 GPU 实现。
* 许多 Python 实现会用到 `NDArray.make`：
```python
def make(shape, strides=None, device=None, handle=None, offset=0):
```
该函数根据给定的 shape、strides、device、handle 和 offset 创建新的 NDArray。如果未指定 `handle`（即不引用已有内存），函数会分配所需内存；如果指定了 `handle`，则不会分配新内存，而是让新 NDArray 指向同一块内存。为了提高效率，尽可能多的函数不应分配新内存，因此许多情况下都应使用此调用。
* NDArray 提供 `.numpy()` 调用，可将数组转换为 NumPy。它与 `numpy_device` 后端不同：该调用会创建一个与给定 NDArray 等价的真正 `numpy.ndarray`，维度和 shape 相同，但 strides 不一定相同（Pybind11 可能会为返回的矩阵重新分配内存，从而改变步长）。

## Part 1：Python 数组操作

首先，请在 [`ndarray.py`](https://github.com/dlsyscourse/hw3/blob/main/python/needle/backend_ndarray/ndarray.py) 文件中实现以下函数：

- `reshape()`
- `permute()`
- `broadcast_to()`
- `__getitem__()`

这些函数的输入和输出都在函数桩的 docstring 中说明。需要特别强调：这些函数都不应重新分配内存，而应返回与 `self` 共享内存的 NDArray，并通过调整 shape、strides 等信息实现相应变换。

可以参考课程幻灯片中关于转置、广播和切片操作符的提示。

注意，`__getitem__()` 与 NumPy 不同，不会改变数组的维度数。例如，对于二维 NDArray `A`，`A[2,2]` 仍会返回二维数组，包含一行一列；`A[:4,2]` 会返回包含 4 行 1 列的二维 NDArray。

`__getitem__()` 开头提供的代码已经将每个索引都转换成了 `slice`，其 `start`、`stop` 和 `step` 均为显式的非负值（包括整数索引和负索引，例如 `A[-1, :]` 或 `A[:, -3:]`），因此实现时只需处理这种规范化后的形式。

本部分可以使用 [`ndarray_backend_numpy.py`](https://github.com/dlsyscourse/hw3/blob/main/python/needle/backend_ndarray/ndarray_backend_numpy.py) 中的所有代码。也可以对照相应的 NumPy 操作结果；测试用例中应该能看到这些结果。

完成后，请运行并提交以下测试。下面的测试会覆盖上述四个函数；你可以随着每个函数的实现逐步尝试通过更多测试。

In [ ]:
!python3 -m pytest -v -k "(permute or reshape or broadcast or getitem) and cpu and not compact"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_python_ops"

## Part 2：CPU 后端——Compact 与 setitem

请在 `ndarray_backend_cpu.cc` 中实现以下函数：
* `Compact()`
* `EwiseSetitem()`
* `ScalarSetitem()`

为了理解为什么它们属于同一类，先来看 `Compact()` 函数。若矩阵在内存中按“行优先”形式连续排列（更准确地说，是推广到高维数组的行优先形式），则称其为 compact：最后一个维度连续排列，其前是倒数第二个维度，依此类推直到第一个维度。在本实现中还要求底层后端数组的总分配大小等于数组本身大小（也就是说，数组数据前后都不能有其他数据；例如 `.offset` 字段必须为 0）。

以三维数组为例，下面展示了 `Compact()` 可能进行的操作。这里的 `shape`、`strides` 和 `offset` 分别是待 compact 矩阵（即 compact 之前）的形状、步长和偏移量。

```c++
cnt = 0;
for (size_t i = 0; i < shape[0]; i++)
    for (size_t j = 0; j < shape[1]; j++)
        for (size_t k = 0; k < shape[2]; k++)
            out[cnt++] = in[offset + strides[0]*i + strides[1]*j + strides[2]*k];
```
也就是说，我们要将基于步长的矩阵表示转换为纯连续的形式。

实现 `Compact()` 的难点是让它适用于任意数量的输入维度。针对固定维度的数组很容易分别编写实现，但通用实现需要模拟“可变数量的循环”。一种方法是维护一个长度等于维度数的索引向量，并在循环中手动递增索引（当索引达到最大值时执行进位操作）。

如果实在难以实现，也可以利用一个事实：我们可能不会要求你处理超过 6 维的矩阵（但 im2col 操作会用到 6 维矩阵）。

#### setitem 与 Compact 的关系

setitem 功能看起来与 `Compact()` 很不相同，但实际上关系密切。设置对象中元素时会调用 Python 函数 `__setitem()__`，例如：
```python
A[::2,4:5,9] = 0 # 或 = some_other_array
```
应该如何实现？上面的 `__getitem()__` 调用已经实现了无需复制、只调整步长即可获取矩阵子集的方法。但要如何设置这个数组中的元素？本作业中的其他许多操作都会在写入输出数组前调用 `.compact()`，但这里不能这样做：调用 `.compact()` 会把子集复制到新内存中，而 `__setitem__()` 的目的正是修改已有内存。

仔细思考后会发现，这与 `.compact()` 很相似，只是方向相反。如果要将一个（本身已经 compact 的）右侧矩阵赋值给 `__getitem()__` 的结果，那么这里的 `shape`、`strides` 和 `offset` 应对应**输出**矩阵。这样 setitem 操作就可以写成：

```c++
cnt = 0;
for (size_t i = 0; i < shape[0]; i++)
    for (size_t j = 0; j < shape[1]; j++)
        for (size_t k = 0; k < shape[2]; k++)
            out[offset + strides[0]*i + strides[1]*j + strides[2]*k] = in[cnt++]; // 或 "= val;"
```

由于两者相似，如果将索引逻辑模块化实现，就能在 `Compact()`、`EwiseSetitem()` 和 `ScalarSetitem()` 中复用。

**注意：**运行测试前，请先运行下面的 `make` 单元格，以重新构建修改后的 C++ 代码。

In [ ]:
!make

In [ ]:
!python3 -m pytest -v -k "(compact or setitem) and cpu"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_compact_setitem"

## Part 3：CPU 后端——逐元素和标量操作

请在 `ndarray_backend_cpu.cc` 中实现以下函数：

* `EwiseMul()`、`ScalarMul()`
* `EwiseDiv()`、`ScalarDiv()`
* `ScalarPower()`
* `EwiseMaximum()`、`ScalarMaximum()`
* `EwiseEq()`、`ScalarEq()`
* `EwiseGe()`、`ScalarGe()`
* `EwiseLog()`
* `EwiseExp()`
* `EwiseTanh()`

可以参考已有的 `EwiseAdd()` 和 `ScalarAdd()` 函数，以及 NDArray 对这些函数的调用方式，了解所需的函数格式。

与此处列出的其他函数不同，我们没有为每个函数提供函数桩。这是因为虽然可以分别朴素地实现每个函数，但会造成大量重复代码。你可以使用 C++ 模板或宏来精简实现（它们只会在内部使用，不会暴露给外部接口）。

**将函数暴露给 Python：**`ndarray_backend_cpu.cc` 底部的 `PYBIND11_MODULE` 代码块中包含了这些函数的绑定，但目前已注释掉（例如 `// m.def("ewise_mul", EwiseMul);`）。实现每个函数后请取消对应绑定的注释，然后重新运行 `make`。如果忘记这一步，测试会报错：`AttributeError: module 'needle.backend_ndarray.ndarray_backend_cpu' has no attribute 'ewise_mul'`。Part 4 和 Part 5 中的归约、矩阵乘法绑定也需要进行同样处理。

In [ ]:
!make

In [ ]:
!python3 -m pytest -v -k "(ewise_fn or ewise_max or log or exp or tanh or (scalar and not setitem)) and cpu"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_ops"

## Part 4：CPU 后端——归约

请在 `ndarray_backend_cpu.cc` 中实现以下函数：

* `ReduceMax()`
* `ReduceSum()`

一般来说，NDArray 的 `.max()` 和 `.sum()` 归约函数会根据 `axis` 参数，沿指定轴求最大值或总和；若 `axis=None`，则对整个数组操作。我们不支持将多个轴组成的集合传给 `axis`，不过如果你想扩展，实现起来也不难；只是作业要求实现的接口不包含此功能。

即使数组是 compact 的，沿某个轴求和也可能有些棘手。因此，Python 端的实现会先将最后一个轴置换到待归约轴的位置（NDArray 中的 `reduce_view_out()` 函数会完成此操作），然后将数组 compact 化。因此，在 C++ 中实现 `ReduceMax()` 和 `ReduceSum()` 时，可以假设输入和输出数组在内存中连续排列，并且只需对传入 C++ 函数的 `reduce_size` 个连续元素执行归约。

记得取消 `ndarray_backend_cpu.cc` 底部 `reduce_max` 和 `reduce_sum` 绑定的注释。

In [ ]:
!make

In [ ]:
!python3 -m pytest -v -k "reduce and cpu"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_reductions"

## Part 5：CPU 后端——矩阵乘法

请在 `ndarray_backend_cpu.cc` 中实现以下函数：

* `Matmul()`
* `MatmulTiled()`
* `AlignedDot()`

`Matmul()` 可以使用朴素的三重嵌套循环实现矩阵乘法。`MatmulTiled()` 则需要对以分块形式存储的矩阵执行相同运算，即将矩阵视为连续的四维数组：
```c++
float[M/TILE][N/TILE][TILE][TILE];
```

计算矩阵乘法时，请确保先将输出矩阵初始化为 0，再填入正确结果。

当矩阵乘法的所有维度大小都能被 `TILE` 整除时，Python 的 `__matmul__` 代码会负责转换为分块形式，因此你只需按这种布局实现乘法。为了提高效率，实现后请使用 `AlignedDot()`；它能帮助编译器有效利用向量运算和缓存。输出矩阵也会采用上述分块形式，Python 后端会负责将其转换为普通二维数组。

记得取消 `ndarray_backend_cpu.cc` 底部 `matmul` 和 `matmul_tiled` 绑定的注释。下方测试包含直接调用 `MatmulTiled()` 的 `test_matmul_tiled`，会测试多个分块网格（包括非方形网格）；mugrade 提交也会运行该测试，因此请确保本地通过。

若想让分块版本获得尽可能大的加速，可以在 Colab 中使用 clang，而非 gcc。运行下方第二个单元格（而不是普通的 `!make`）即可安装 clang（Colab 未预装，需要约一分钟），并删除旧构建产物，因为 CMake 会记住首次构建时使用的编译器。之后的 `make` 会继续使用 clang，直到运行 `make clean`。

In [ ]:
!make

In [ ]:
!apt-get -qq update 2> /dev/null && apt-get -qq install -y clang > /dev/null && make clean && CXX=clang++ make

In [ ]:
!python3 -m pytest -v -k "matmul and (cpu or tiled)"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_matmul"

## 切换到 GPU 运行环境

剩余部分需要使用 GPU。请通过 **Runtime → Change runtime type → T4 GPU**（或其他 GPU）进行切换。切换后会启动一台全新的机器，因此请按以下步骤操作：

1. 重新运行 notebook 顶部的设置单元格（挂载 Drive、进入 hw3 目录并设置 `MY_API_KEY`）。代码保存在 Drive 中，因此不会丢失。
2. 运行下面的单元格，从头开始重新构建。需要干净构建，以便 CMake 检测 CUDA，并为当前机器编译两个后端。请确认输出中包含 `Found cuda, building cuda backend`。

如果后续 CUDA 测试显示为 `SKIPPED`，说明 CUDA 后端未能构建或导入；跳过提示会说明原因。

In [ ]:
!make clean && make

## Part 6：CUDA 后端——Compact 与 setitem

请在 `ndarray_backend_cuda.cu` 中实现以下函数：
* `Compact()`
* `EwiseSetitem()`
* `ScalarSetitem()`

本部分需要在 CUDA 后端实现 compact 和 setitem 调用。整体思路与 C++ 版本相似，不过具体差异可能取决于你对 C++ 函数的实现方式。下面重点说明 C++ 和 CUDA 实现之间的几个差别。

与 CUDA 后端中已有的示例函数一样，上述每个操作都需要实现两个函数：供 Python 调用的基础函数，以及实际执行计算的 CUDA kernel。大多数情况下，`ndarray_backend_cuda.cu` 只提供“基础”函数的声明，你需要自行定义并实现 kernel。不过，为帮助你理解其工作方式，`Compact()` 的完整函数实现以及 `CompactKernel()` 的函数声明已经提供。

你可能会注意到这里使用了 `CudaVec` 结构体，它用于传递 shape/stride 参数。C++ 版本使用 STL 的 `std::vector` 保存这些输入（基础 `Compact()` 函数也是如此），但 CUDA kernel 无法操作 STL vector。此外，虽然可以把 vector 转成普通 CUDA 数组，但这会很繁琐：需要调用 `cudaMalloc()`、传递整数指针，并在调用后释放内存。数组底层数据确实需要这样的内存管理，但仅为传递一个长度可变的小型 shape/stride 元组而这样做就有些过度了。解决方法是定义一个结构体，预留足够多的维度空间，并将实际 shape/stride 数据存入字段的前几个位置。提供的 `CudaVec` 结构体和 `VecToCuda()` 函数已经完成了这项工作；需要向 kernel 传递 shape/strides 时可直接使用。

C++ 和 CUDA 版 `Compact()` 的另一个重要差别在于：C++ 通常会按顺序遍历非 compact 数组的元素，并利用遍历顺序优化紧凑数组与非紧凑数组之间的索引计算。CUDA 中无法这样做，因此需要实现从 compact 数组中的索引直接映射到步长数组中的索引。

与前面一样，建议以便于复用的方式编写代码，让 `Compact()` 和 `Setitem()` 调用可以共享实现。另请注意，若要在 kernel 代码中调用另一个（非 kernel）函数，需要将其声明为 `__device__` 函数。在 CUDA 中，`__global__` 用于定义由 CPU 主机调用、在 GPU 上运行的函数；`__device__` 用于定义只在 GPU 上运行、并由其他 GPU 函数调用的函数。

In [ ]:
!make

In [ ]:
!python3 -m pytest -v -k "(compact or setitem) and cuda"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_compact_setitem"

## Part 7：CUDA 后端——逐元素和标量操作

请在 `ndarray_backend_cuda.cu` 中实现以下函数：

* `EwiseMul()`、`ScalarMul()`
* `EwiseDiv()`、`ScalarDiv()`
* `ScalarPower()`
* `EwiseMaximum()`、`ScalarMaximum()`
* `EwiseEq()`、`ScalarEq()`
* `EwiseGe()`、`ScalarGe()`
* `EwiseLog()`
* `EwiseExp()`
* `EwiseTanh()`

同样，我们没有提供这些函数的声明；你可以使用 C++ 模板或宏精简实现。与 Part 3 相同，实现每个函数后请取消 `ndarray_backend_cuda.cu` 底部对应 `m.def(...)` 绑定的注释，然后重新构建。

In [ ]:
!make

In [ ]:
!python3 -m pytest -v -k "(ewise_fn or ewise_max or log or exp or tanh or (scalar and not setitem)) and cuda"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_ops"

## Part 8：CUDA 后端——归约

请在 `ndarray_backend_cuda.cu` 中实现以下函数：

* `ReduceMax()`
* `ReduceSum()`

可以使用相对简单的方法：为每个单独的归约任务分配一个 CUDA 线程。例如，对一个 100 × 20 的数组沿第二个维度归约，可以分配 100 个线程，每个线程分别处理一个长度为 20 的数组。这种方式对于 `.max(axis=None)` 特别低效，但目前无需担心。如果想实现更高性能的方案，可以采用分层机制：先分别对较小的数据块求和，再通过第二阶段函数对这些归约结果继续归约。不过，通过测试不需要采用这种方式。

记得取消 `ndarray_backend_cuda.cu` 底部 `reduce_max` 和 `reduce_sum` 绑定的注释。

In [ ]:
!make

In [ ]:
!python3 -m pytest -v -k "reduce and cuda"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_reductions"

## Part 9：CUDA 后端——矩阵乘法

请在 `ndarray_backend_cuda.cu` 中实现以下函数：

* `Matmul()`

最后一个练习是在 GPU 上实现矩阵乘法。实现方式大致可以参考课堂讲解。使用相当朴素的代码也能通过测试，例如为矩阵中每个 `(i,j)` 位置分配一个线程来完成乘法；但若要获得优于 CPU 版本的速度，需要像课堂讲解的那样，协同取数并使用线程块共享内存和寄存器分块。请尝试这些方法，并观察实现相较于 C++（或 NumPy）后端能快多少。

记得取消 `ndarray_backend_cuda.cu` 底部 `matmul` 绑定的注释。

In [ ]:
!make

In [ ]:
!python3 -m pytest -v -k "matmul and cuda"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_matmul"

## 提交代码

上方的 `mugrade submit` 单元格会对每个部分进行评分。完成并提交所有部分后，还需要将代码压缩包上传到 mugrade 上的 [Homework 3 作业页面](https://mugrade.org/assignments/39)。

下方单元格会运行 [`make_archive.py`](https://github.com/dlsyscourse/hw3/blob/main/make_archive.py)，将 `python/` 和 `src/`（本次作业中需要修改的所有文件）打包为 `hw3.tar.gz`，并排除 `build/` 和 `.so` 等构建输出文件，以免压缩包过大而无法上传。无需包含 HW1/HW2 的代码。如果下载没有自动开始，压缩包也会保存在 Drive 的 `hw3` 文件夹中。

In [ ]:
!python3 make_archive.py
from google.colab import files
files.download("hw3.tar.gz")